# Proyecto Final – Analítica de Negocios  
## Predicción de enfermedad cardíaca mediante Machine Learning  
### Random Forest Classifier y Redes Neuronales MLPRegressor

**Dataset:** UCI Heart Disease Dataset – `heart_disease_combined.csv`  
**Curso:** Analítica de Negocios – Universidad EAFIT  
**Año:** 2026  
**Integrantes:** Samuel David Giraldo Gil y Jacobo Hinestroza Escobar  

---

Este notebook unifica el análisis completo del proyecto final. Está organizado como portafolio académico y técnico siguiendo la rúbrica del challenge: introducción, definición del problema, recopilación y preparación de datos, definición de modelos, método de trabajo, métricas, validación, discusión, consideraciones éticas, conclusiones, recomendaciones y bibliografía.

## Tabla de contenido

1. Introducción  
2. Definición del problema  
3. Recopilación de datos y descripción del dataset  
4. Diccionario de variables  
5. Preparación y limpieza de datos  
6. Análisis exploratorio inicial  
7. Modelo 1: Random Forest Classifier  
8. Modelo 2: MLPRegressor para pronóstico de colesterol  
9. Modelo 3: MLPRegressor para pronóstico de frecuencia cardíaca máxima  
10. Comparación de resultados  
11. Discusión de resultados  
12. Consideraciones éticas y prácticas  
13. Conclusiones  
14. Recomendaciones  
15. Bibliografía  
16. Anexo: estructura sugerida para GitHub

# 1. Introducción

La analítica de negocios permite transformar datos en información útil para apoyar la toma de decisiones. En el sector salud, esta capacidad es especialmente relevante porque las decisiones oportunas pueden influir en la detección temprana de enfermedades, la priorización de pacientes, la asignación de recursos clínicos y la prevención de complicaciones.

Este proyecto aplica modelos de Machine Learning al **UCI Heart Disease Dataset**, una base de datos clínica ampliamente utilizada en investigación médica y aprendizaje automático. El objetivo no es reemplazar el criterio médico, sino evaluar cómo los modelos analíticos pueden apoyar la identificación de pacientes con posible enfermedad cardíaca y explorar la posibilidad de pronosticar variables clínicas numéricas relevantes.

Para cumplir con la rúbrica del challenge se desarrollaron dos enfoques principales:

1. **Modelo de clasificación:** Random Forest Classifier para predecir la variable `target`, donde 0 indica ausencia de enfermedad cardíaca y 1 indica presencia de enfermedad cardíaca.
2. **Modelo de pronóstico:** MLPRegressor para predecir variables clínicas numéricas. Primero se evaluó el colesterol (`chol`) y posteriormente se mejoró el enfoque utilizando frecuencia cardíaca máxima alcanzada (`thalach`).

El análisis combina código ejecutable, visualizaciones, interpretación de métricas y discusión académica sobre los resultados.

# 2. Definición del problema

El problema de negocio se centra en la **detección temprana de enfermedad cardíaca** a partir de múltiples variables clínicas. En un entorno médico real, interpretar de forma simultánea edad, presión arterial, colesterol, frecuencia cardíaca máxima, dolor torácico, resultados de electrocardiograma y otras variables puede ser complejo.

Un diagnóstico tardío puede aumentar costos de atención, generar complicaciones evitables, reducir la oportunidad de intervención médica y afectar la calidad de vida del paciente. Por esta razón, un modelo de Machine Learning puede ser útil como herramienta de apoyo para identificar patrones de riesgo y priorizar pacientes.

## Preguntas analíticas del proyecto

- ¿Qué variables clínicas ayudan más a detectar presencia de enfermedad cardíaca?
- ¿Qué desempeño alcanza un modelo Random Forest para clasificar pacientes con y sin enfermedad cardíaca?
- ¿Puede una red neuronal MLPRegressor pronosticar una variable clínica numérica relevante?
- ¿Qué limitaciones prácticas y éticas deben considerarse antes de utilizar modelos predictivos en salud?

## Objetivo general

Construir y evaluar modelos de Machine Learning que permitan clasificar la presencia de enfermedad cardíaca y pronosticar variables clínicas numéricas, utilizando un dataset médico real y métricas apropiadas para cada tipo de problema.

## Objetivos específicos

- Describir y preparar el dataset clínico.
- Implementar un modelo Random Forest para clasificación binaria.
- Evaluar el desempeño mediante accuracy, precision, recall, F1, matriz de confusión y AUC.
- Implementar modelos MLPRegressor para pronóstico.
- Comparar el desempeño del MLP contra un modelo base.
- Interpretar resultados y extraer hallazgos útiles para la toma de decisiones.

# 3. Recopilación de datos y descripción del dataset

El dataset utilizado es `heart_disease_combined.csv`, correspondiente a la versión combinada del UCI Heart Disease Dataset. Esta versión integra registros clínicos de pacientes provenientes de varias instituciones médicas internacionales.

El archivo contiene **920 registros** y **15 columnas**, incluyendo variables clínicas, diagnósticas y una variable de fuente institucional (`source`).

In [ ]:
# ============================================
# 3.1 Librerías necesarias
# ============================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPRegressor

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)

In [ ]:
# ============================================
# 3.2 Carga del dataset
# ============================================

# En Google Colab, cargue el archivo heart_disease_combined.csv en /content/
# Si el archivo está en otra ruta, modifique DATA_PATH.
DATA_PATH = "/content/heart_disease_combined.csv"

# Fallback para ejecución local/sandbox.
if not os.path.exists(DATA_PATH):
    DATA_PATH = "/mnt/data/heart_disease_combined.csv"

df = pd.read_csv(DATA_PATH)

print("Dimensiones del dataset:", df.shape)
display(df.head())

In [ ]:
# Información general del dataset
print("Columnas del dataset:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.dtypes)

print("\nValores nulos por columna:")
print(df.isnull().sum())

print("\nDistribución de la variable target:")
print(df["target"].value_counts())
print(df["target"].value_counts(normalize=True).round(4))

## 3.3 Resumen del dataset

| Elemento | Valor |
|---|---|
| Archivo utilizado | `heart_disease_combined.csv` |
| Registros | 920 |
| Columnas totales | 15 |
| Variable objetivo principal | `target` |
| Tipo de problema principal | Clasificación binaria |
| Clase 0 | Sin enfermedad cardíaca |
| Clase 1 | Con enfermedad cardíaca |
| Fuente | UCI Heart Disease Dataset / versión combinada en Kaggle |

El dataset también contiene la columna `source`, que identifica la institución médica de origen del registro. Para Random Forest se excluyó `source`, ya que el objetivo era clasificar con variables clínicas del paciente y no con la institución de procedencia.

# 4. Diccionario de variables

El archivo contiene exactamente 15 columnas. Las variables clínicas se utilizaron de forma diferenciada según el modelo.

| Variable | Tipo | Descripción | Rol en el proyecto |
|---|---|---|---|
| `age` | Numérica | Edad del paciente | Predictora |
| `sex` | Categórica codificada | Sexo: 1 = hombre, 0 = mujer | Predictora |
| `cp` | Categórica codificada | Tipo de dolor torácico | Predictora |
| `trestbps` | Numérica | Presión arterial en reposo | Predictora |
| `chol` | Numérica | Colesterol sérico | Predictora RF / objetivo MLP inicial |
| `fbs` | Binaria | Glucosa en ayunas > 120 mg/dl | Predictora |
| `restecg` | Categórica codificada | ECG en reposo | Predictora |
| `thalach` | Numérica | Frecuencia cardíaca máxima alcanzada | Predictora RF / objetivo MLP final |
| `exang` | Binaria | Angina inducida por ejercicio | Predictora |
| `oldpeak` | Numérica | Depresión ST inducida por ejercicio | Predictora |
| `slope` | Categórica codificada | Pendiente del segmento ST | Predictora |
| `ca` | Numérica/codificada | Número de vasos principales coloreados | Predictora |
| `thal` | Categórica codificada | Talasemia | Predictora |
| `target` | Binaria | 0 = no enfermedad, 1 = enfermedad | Objetivo Random Forest |
| `source` | Categórica | Institución de origen | Metadato / posible predictora en MLP |

# 5. Preparación y limpieza de datos

La preparación de datos fue diferente para cada modelo porque los objetivos son distintos:

- En **Random Forest**, la variable objetivo fue `target`. Se usaron 13 variables clínicas predictoras y se excluyó `source`.
- En **MLPRegressor**, se trabajó con variables objetivo numéricas. Primero se intentó predecir `chol` y luego se cambió a `thalach`.

Para evitar pérdida masiva de información, se aplicó **imputación por mediana** en valores faltantes de variables numéricas. La mediana es adecuada para variables clínicas porque es menos sensible a valores extremos que la media.

En el caso de redes neuronales, además de imputar, se aplicó **StandardScaler**, ya que las redes neuronales son sensibles a la escala de las variables. Para variables categóricas tipo objeto, como `source`, se aplicó **OneHotEncoder**.

# 6. Análisis exploratorio inicial

Antes de modelar se revisaron dimensiones, nulos, distribución de la variable objetivo y algunas distribuciones clínicas. Esto permite entender la estructura del dataset y detectar posibles problemas de calidad de datos.

In [ ]:
# Distribución de target
plt.figure(figsize=(5,4))
sns.countplot(data=df, x="target")
plt.title("Distribución de la variable objetivo target")
plt.xlabel("target (0 = No enfermedad, 1 = Enfermedad)")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

In [ ]:
# Distribuciones de variables numéricas principales
num_cols = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]

for col in num_cols:
    plt.figure(figsize=(6,4))
    sns.histplot(df[col], kde=True)
    plt.title(f"Distribución de {col}")
    plt.xlabel(col)
    plt.ylabel("Frecuencia")
    plt.tight_layout()
    plt.show()

In [ ]:
# Matriz de correlación de variables numéricas
plt.figure(figsize=(9,7))
corr = df.select_dtypes(include=["int64", "float64"]).corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Matriz de correlación de variables numéricas")
plt.tight_layout()
plt.show()

# 7. Modelo 1: Random Forest Classifier

## 7.1 Justificación del modelo

Random Forest es un modelo supervisado de clasificación basado en la combinación de múltiples árboles de decisión. Cada árbol aprende patrones a partir de diferentes subconjuntos de datos y variables. La predicción final se obtiene mediante votación mayoritaria.

Este modelo fue elegido porque:

- Maneja bien relaciones no lineales.
- Es robusto frente a ruido y sobreajuste.
- Permite calcular importancia de variables.
- Es adecuado para clasificación binaria.
- Tiene buena interpretabilidad relativa frente a modelos más complejos.

## 7.2 Variable objetivo

La variable objetivo fue `target`:

- `0` = no enfermedad cardíaca.
- `1` = presencia de enfermedad cardíaca.

In [ ]:
# ============================================
# 7.3 Preparación específica para Random Forest
# ============================================

# Variables categóricas codificadas numéricamente
X_categorical = df[[
    "sex", "cp", "fbs", "restecg", "exang", "slope", "thal"
]].copy()

# Variables numéricas
X_numerical = df[[
    "age", "trestbps", "chol", "thalach", "oldpeak", "ca"
]].copy()

# Variable objetivo
y_rf = df["target"].copy()

# Unir variables predictoras
X_rf = pd.concat([
    X_categorical.reset_index(drop=True),
    X_numerical.reset_index(drop=True)
], axis=1)

print("Valores nulos antes de imputación:")
display(X_rf.isnull().sum())

# Imputación por mediana
X_rf = X_rf.fillna(X_rf.median())

print("\nValores nulos después de imputación:")
display(X_rf.isnull().sum())

print("\nDimensión final X_rf:", X_rf.shape)
print("Dimensión final y_rf:", y_rf.shape)

In [ ]:
# División 80/20 con estratificación
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X_rf,
    y_rf,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_rf
)

print("Entrenamiento:", X_train_rf.shape, y_train_rf.shape)
print("Prueba:", X_test_rf.shape, y_test_rf.shape)

print("\nDistribución target en train:")
print(y_train_rf.value_counts(normalize=True).round(4))

print("\nDistribución target en test:")
print(y_test_rf.value_counts(normalize=True).round(4))

## 7.4 Hiperparámetros y método de selección

Se definió un GridSearchCV con validación cruzada de 5 particiones. La métrica de optimización fue **F1 Score**, porque en un problema médico no basta con maximizar accuracy: también se debe equilibrar precisión y sensibilidad.

| Hiperparámetro | Valores evaluados |
|---|---|
| `n_estimators` | 100, 200, 300 |
| `max_depth` | None, 5, 10, 15 |
| `min_samples_split` | 2, 5, 10 |
| `min_samples_leaf` | 1, 2, 4 |
| `max_features` | sqrt, log2 |

En la ejecución del proyecto, la mejor configuración encontrada fue:

```python
{
    'max_depth': None,
    'max_features': 'sqrt',
    'min_samples_leaf': 2,
    'min_samples_split': 10,
    'n_estimators': 200
}
```

Mejor F1 Score en validación cruzada: **0.8438**.

In [ ]:
# ============================================
# 7.5 GridSearchCV
# ============================================

# Para ahorrar tiempo en Colab, el notebook usa por defecto los mejores hiperparámetros ya encontrados.
# Si desea repetir la búsqueda completa, cambie EJECUTAR_GRID_SEARCH a True.
EJECUTAR_GRID_SEARCH = False

Grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10, 15],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

if EJECUTAR_GRID_SEARCH:
    modRF = RandomForestClassifier(random_state=RANDOM_STATE)
    Grid_S = GridSearchCV(
        estimator=modRF,
        param_grid=Grid,
        cv=5,
        scoring="f1",
        n_jobs=-1,
        refit=True
    )
    Grid_S.fit(X_train_rf, y_train_rf)
    best_rf = Grid_S.best_estimator_
    print("Mejores hiperparámetros encontrados:")
    print(Grid_S.best_params_)
    print(f"Mejor F1 Score CV: {Grid_S.best_score_:.4f}")
else:
    best_rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        max_features="sqrt",
        min_samples_leaf=2,
        min_samples_split=10,
        random_state=RANDOM_STATE
    )
    best_rf.fit(X_train_rf, y_train_rf)
    print("Modelo entrenado con los mejores hiperparámetros encontrados previamente.")

In [ ]:
# ============================================
# 7.6 Evaluación del Random Forest
# ============================================

y_pred_rf = best_rf.predict(X_test_rf)
y_proba_rf = best_rf.predict_proba(X_test_rf)[:, 1]

acc_rf = accuracy_score(y_test_rf, y_pred_rf)
prec_rf = precision_score(y_test_rf, y_pred_rf)
rec_rf = recall_score(y_test_rf, y_pred_rf)
f1_rf = f1_score(y_test_rf, y_pred_rf)
auc_rf = roc_auc_score(y_test_rf, y_proba_rf)

cm_rf = confusion_matrix(y_test_rf, y_pred_rf)

print("RESULTADOS RANDOM FOREST")
print(f"Accuracy     : {acc_rf:.4f}")
print(f"Precision    : {prec_rf:.4f}")
print(f"Recall       : {rec_rf:.4f}")
print(f"F1 Score     : {f1_rf:.4f}")
print(f"AUC          : {auc_rf:.4f}")

print("\nMatriz de confusión:")
print(cm_rf)

print("\nReporte de clasificación:")
print(classification_report(y_test_rf, y_pred_rf))

## 7.7 Resultados obtenidos en la ejecución del proyecto

Los resultados finales del Random Forest fueron:

| Métrica | Resultado |
|---|---:|
| Accuracy | 0.8424 |
| Precision clase 1 | 0.8411 |
| Recall / Sensibilidad | 0.8824 |
| Especificidad | 0.7927 |
| Valor predictivo negativo | 0.8442 |
| F1 Score | 0.8612 |
| Tasa de error | 0.1576 |
| AUC | 0.9201 |

La matriz de confusión obtenida fue:

|  | Predicho 0 | Predicho 1 |
|---|---:|---:|
| Real 0 | 65 | 17 |
| Real 1 | 12 | 90 |

Interpretación: el modelo detectó correctamente 90 pacientes enfermos y 65 pacientes sanos. El punto más sensible desde el punto de vista clínico son los 12 falsos negativos, ya que representan pacientes con enfermedad cardíaca clasificados como sanos.

In [ ]:
# Visualización matriz de confusión
plt.figure(figsize=(5,4))
sns.heatmap(
    cm_rf,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Predicho 0", "Predicho 1"],
    yticklabels=["Real 0", "Real 1"]
)
plt.title("Matriz de confusión - Random Forest")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.tight_layout()
plt.show()

In [ ]:
# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test_rf, y_proba_rf)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"Random Forest (AUC = {auc_rf:.4f})")
plt.plot([0,1], [0,1], linestyle="--", label="Clasificación aleatoria")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC - Random Forest")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Importancia de variables por impureza
importances_rf = pd.Series(
    best_rf.feature_importances_,
    index=X_rf.columns
).sort_values(ascending=False)

print("Importancia de variables:")
display(importances_rf)

plt.figure(figsize=(8,5))
importances_rf.head(10).sort_values().plot(kind="barh")
plt.title("Top 10 variables más importantes - Random Forest")
plt.xlabel("Importancia")
plt.ylabel("Variable")
plt.tight_layout()
plt.show()

## 7.8 Importancia de variables

Las variables más importantes por impureza fueron:

| Variable | Importancia |
|---|---:|
| `cp` | 0.182860 |
| `chol` | 0.128346 |
| `thalach` | 0.119639 |
| `oldpeak` | 0.119571 |
| `age` | 0.112165 |
| `exang` | 0.103233 |

El resultado es clínicamente razonable: el tipo de dolor torácico, el colesterol, la frecuencia cardíaca máxima, la depresión ST, la edad y la angina inducida por ejercicio son variables vinculadas con el riesgo cardiovascular.

In [ ]:
# Permutation importance: validación adicional de importancia
pi = permutation_importance(
    best_rf,
    X_test_rf,
    y_test_rf,
    n_repeats=20,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

perm_importance_rf = pd.Series(
    pi.importances_mean,
    index=X_rf.columns
).sort_values(ascending=False)

print("Permutation importance:")
display(perm_importance_rf.head(10))

# 8. Modelo 2: MLPRegressor para pronóstico de colesterol (`chol`)

## 8.1 Objetivo

Para cumplir el componente de pronóstico de la rúbrica, se implementó una red neuronal MLPRegressor. El primer intento consistió en predecir la variable `chol`, correspondiente al colesterol sérico del paciente.

Este modelo no clasifica pacientes, sino que estima un valor numérico continuo. Por eso las métricas utilizadas fueron:

- **MAE:** error absoluto medio.
- **RMSE:** raíz del error cuadrático medio.
- **R²:** proporción de variabilidad explicada por el modelo.

## 8.2 Limpieza específica

Se eliminaron registros donde `chol` fuera nulo o menor/igual a cero, porque un colesterol de 0 no es clínicamente válido.

In [ ]:
# ============================================
# 8.3 Preparación del dataset para pronóstico de colesterol
# ============================================

df_chol = df.dropna(subset=["chol"]).copy()
df_chol = df_chol[df_chol["chol"] > 0].copy()

print("Dimensión df_chol:", df_chol.shape)
display(df_chol["chol"].describe())

target_col = "chol"
X_chol = df_chol.drop(columns=[target_col]).copy()
y_chol = df_chol[target_col].copy()

print("X_chol:", X_chol.shape)
print("y_chol:", y_chol.shape)

In [ ]:
# División 60/20/20 para MLP
X_train_full_chol, X_test_chol, y_train_full_chol, y_test_chol = train_test_split(
    X_chol,
    y_chol,
    test_size=0.20,
    random_state=RANDOM_STATE
)

X_train_chol, X_valid_chol, y_train_chol, y_valid_chol = train_test_split(
    X_train_full_chol,
    y_train_full_chol,
    test_size=0.25,
    random_state=RANDOM_STATE
)

print("Train:", X_train_chol.shape)
print("Valid:", X_valid_chol.shape)
print("Test:", X_test_chol.shape)

In [ ]:
# Preprocesador robusto para MLP
try:
    onehot = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
except TypeError:
    onehot = OneHotEncoder(sparse=False, handle_unknown="ignore")

numerical_features = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_features = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", onehot)
])

preprocessor_mlp = ColumnTransformer([
    ("num", numerical_features, make_column_selector(dtype_include=["int64", "float64"])),
    ("cat", categorical_features, make_column_selector(dtype_include=["object", "category"]))
])

In [ ]:
# Modelo MLPRegressor inicial para colesterol
mlp_chol = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=1e-3,
    learning_rate_init=1e-3,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=RANDOM_STATE,
    verbose=False
)

pipe_chol = Pipeline(steps=[
    ("prep", preprocessor_mlp),
    ("model", mlp_chol)
])

pipe_chol.fit(X_train_chol, y_train_chol)

y_pred_chol = pipe_chol.predict(X_test_chol)

mae_chol = mean_absolute_error(y_test_chol, y_pred_chol)
rmse_chol = np.sqrt(mean_squared_error(y_test_chol, y_pred_chol))
r2_chol = r2_score(y_test_chol, y_pred_chol)

# Modelo base: predice siempre el promedio del entrenamiento
y_base_chol = np.full(shape=len(y_test_chol), fill_value=y_train_chol.mean())
mae_base_chol = mean_absolute_error(y_test_chol, y_base_chol)
rmse_base_chol = np.sqrt(mean_squared_error(y_test_chol, y_base_chol))
r2_base_chol = r2_score(y_test_chol, y_base_chol)

print("MODELO BASE - PROMEDIO DEL COLESTEROL")
print(f"MAE  : {mae_base_chol:.4f}")
print(f"RMSE : {rmse_base_chol:.4f}")
print(f"R²   : {r2_base_chol:.4f}")

print("\nMLPRegressor - chol")
print(f"MAE  : {mae_chol:.4f}")
print(f"RMSE : {rmse_chol:.4f}")
print(f"R²   : {r2_chol:.4f}")

## 8.4 Resultado del primer MLP

En la ejecución del proyecto, el modelo MLPRegressor para `chol` obtuvo:

| Modelo | MAE | RMSE | R² |
|---|---:|---:|---:|
| Modelo base - promedio del colesterol | 40.6991 | 52.0709 | -0.0009 |
| MLPRegressor final para `chol` | 43.4284 | 55.3219 | -0.1298 |

El modelo no superó el baseline. El R² negativo indica que la red neuronal explicó peor la variabilidad del colesterol que una predicción constante basada en el promedio. Este resultado es un hallazgo importante: las variables disponibles en el dataset no capturan suficientemente los factores que explican el colesterol, como dieta, genética, medicamentos, peso, actividad física u otros marcadores metabólicos.

In [ ]:
# Gráfico real vs predicho para colesterol
plt.figure(figsize=(6,5))
plt.scatter(y_test_chol, y_pred_chol, alpha=0.7)
plt.plot(
    [y_test_chol.min(), y_test_chol.max()],
    [y_test_chol.min(), y_test_chol.max()],
    linestyle="--"
)
plt.xlabel("Colesterol real")
plt.ylabel("Colesterol predicho")
plt.title("Valores reales vs predichos - MLPRegressor (chol)")
plt.grid(True)
plt.tight_layout()
plt.show()

# 9. Modelo 3: MLPRegressor para pronóstico de frecuencia cardíaca máxima (`thalach`)

## 9.1 Cambio de variable objetivo

Después del bajo desempeño del MLP para colesterol, se realizó un ajuste metodológico: cambiar la variable objetivo a `thalach`, que representa la frecuencia cardíaca máxima alcanzada.

Esta variable tiene mayor coherencia clínica con otras variables del dataset, como edad, dolor torácico, angina inducida por ejercicio, oldpeak y target.

## 9.2 Limpieza específica

Se eliminaron registros donde `thalach` fuera nulo o menor/igual a cero, ya que una frecuencia cardíaca máxima de 0 no es clínicamente válida.

In [ ]:
# ============================================
# 9.3 Preparación del dataset para pronóstico de thalach
# ============================================

df_thalach = df.dropna(subset=["thalach"]).copy()
df_thalach = df_thalach[df_thalach["thalach"] > 0].copy()

print("Dimensión df_thalach:", df_thalach.shape)
display(df_thalach["thalach"].describe())

target_col = "thalach"
X_thalach = df_thalach.drop(columns=[target_col]).copy()
y_thalach = df_thalach[target_col].copy()

print("X_thalach:", X_thalach.shape)
print("y_thalach:", y_thalach.shape)

In [ ]:
# División 60/20/20 para MLP
X_train_full_th, X_test_th, y_train_full_th, y_test_th = train_test_split(
    X_thalach,
    y_thalach,
    test_size=0.20,
    random_state=RANDOM_STATE
)

X_train_th, X_valid_th, y_train_th, y_valid_th = train_test_split(
    X_train_full_th,
    y_train_full_th,
    test_size=0.25,
    random_state=RANDOM_STATE
)

print("Train:", X_train_th.shape)
print("Valid:", X_valid_th.shape)
print("Test:", X_test_th.shape)

In [ ]:
# Modelo MLPRegressor final para thalach
mlp_thalach = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    solver="adam",
    alpha=5e-4,
    learning_rate_init=8e-4,
    max_iter=800,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=30,
    random_state=RANDOM_STATE,
    verbose=False
)

pipe_thalach = Pipeline(steps=[
    ("prep", preprocessor_mlp),
    ("model", mlp_thalach)
])

pipe_thalach.fit(X_train_th, y_train_th)

y_pred_th = pipe_thalach.predict(X_test_th)

mae_th = mean_absolute_error(y_test_th, y_pred_th)
rmse_th = np.sqrt(mean_squared_error(y_test_th, y_pred_th))
r2_th = r2_score(y_test_th, y_pred_th)

# Modelo base: promedio del entrenamiento
y_base_th = np.full(shape=len(y_test_th), fill_value=y_train_th.mean())
mae_base_th = mean_absolute_error(y_test_th, y_base_th)
rmse_base_th = np.sqrt(mean_squared_error(y_test_th, y_base_th))
r2_base_th = r2_score(y_test_th, y_base_th)

print("MODELO BASE - PROMEDIO DE FRECUENCIA CARDÍACA")
print(f"MAE  : {mae_base_th:.4f}")
print(f"RMSE : {rmse_base_th:.4f}")
print(f"R²   : {r2_base_th:.4f}")

print("\nMLPRegressor - thalach")
print(f"MAE  : {mae_th:.4f}")
print(f"RMSE : {rmse_th:.4f}")
print(f"R²   : {r2_th:.4f}")

## 9.4 Resultado del MLP final

En la ejecución del proyecto, el modelo MLPRegressor para `thalach` obtuvo:

| Modelo | MAE | RMSE | R² |
|---|---:|---:|---:|
| Modelo base - promedio frecuencia cardíaca | 21.5843 | 26.2297 | -0.0004 |
| MLPRegressor final (`thalach`) | 18.6394 | 23.7781 | 0.1779 |

El MLP final sí superó el modelo base. Esto demuestra que el cambio de variable objetivo fue metodológicamente adecuado. La red neuronal logró aprender parte de la variabilidad de la frecuencia cardíaca máxima alcanzada, aunque el R² indica que todavía existe variabilidad no explicada por las variables disponibles.

In [ ]:
# Gráfico real vs predicho para thalach
plt.figure(figsize=(6,5))
plt.scatter(y_test_th, y_pred_th, alpha=0.7)
plt.plot(
    [y_test_th.min(), y_test_th.max()],
    [y_test_th.min(), y_test_th.max()],
    linestyle="--"
)
plt.xlabel("Frecuencia cardíaca real")
plt.ylabel("Frecuencia cardíaca predicha")
plt.title("Valores reales vs predichos - MLPRegressor (thalach)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Comparación gráfica baseline vs MLP para thalach
metricas_th = pd.DataFrame({
    "Modelo": ["Baseline", "MLPRegressor"],
    "MAE": [mae_base_th, mae_th],
    "RMSE": [rmse_base_th, rmse_th],
    "R2": [r2_base_th, r2_th]
})

display(metricas_th)

plt.figure(figsize=(7,4))
x = np.arange(len(metricas_th["Modelo"]))
width = 0.35

plt.bar(x - width/2, metricas_th["MAE"], width, label="MAE")
plt.bar(x + width/2, metricas_th["RMSE"], width, label="RMSE")
plt.xticks(x, metricas_th["Modelo"])
plt.title("Comparación de errores - Baseline vs MLPRegressor")
plt.ylabel("Error")
plt.legend()
plt.tight_layout()
plt.show()

# 10. Comparación de resultados

| Modelo | Tipo de problema | Variable objetivo | Métricas principales | Conclusión |
|---|---|---|---|---|
| Random Forest Classifier | Clasificación | `target` | Accuracy 0.8424, F1 0.8612, Recall 0.8824, AUC 0.9201 | Modelo más sólido del proyecto |
| MLPRegressor inicial | Pronóstico | `chol` | MAE 43.4284, RMSE 55.3219, R² -0.1298 | No superó el baseline |
| MLPRegressor final | Pronóstico | `thalach` | MAE 18.6394, RMSE 23.7781, R² 0.1779 | Superó el baseline y mejoró el enfoque |

El mejor modelo general fue Random Forest, porque obtuvo alto desempeño en clasificación y permitió interpretar variables importantes. El MLPRegressor aportó un hallazgo metodológico relevante: la selección de la variable objetivo tiene un impacto directo sobre el rendimiento predictivo.

In [ ]:
# Tabla resumen con resultados reportados del proyecto
resumen_modelos = pd.DataFrame({
    "Modelo": [
        "Random Forest",
        "MLPRegressor chol",
        "MLPRegressor thalach"
    ],
    "Tipo": [
        "Clasificación",
        "Pronóstico",
        "Pronóstico"
    ],
    "Variable objetivo": [
        "target",
        "chol",
        "thalach"
    ],
    "Métrica principal": [
        "F1 = 0.8612 | AUC = 0.9201",
        "RMSE = 55.3219 | R² = -0.1298",
        "RMSE = 23.7781 | R² = 0.1779"
    ],
    "Interpretación": [
        "Modelo más sólido para clasificar enfermedad cardíaca",
        "No superó el modelo base",
        "Mejoró al cambiar la variable objetivo"
    ]
})

display(resumen_modelos)

# 11. Discusión de resultados

Los resultados obtenidos muestran una diferencia clara entre el desempeño del modelo de clasificación y los modelos de pronóstico. El Random Forest Classifier logró un rendimiento sólido al identificar pacientes con y sin enfermedad cardíaca, mientras que el MLPRegressor tuvo un comportamiento más variable dependiendo de la variable objetivo seleccionada. Esta diferencia es relevante porque demuestra que no basta con aplicar modelos complejos; también es necesario evaluar si la estructura del dataset contiene información suficiente para explicar la variable que se quiere predecir.

En el caso del Random Forest, el modelo alcanzó un accuracy de 0.8424, un F1 Score de 0.8612, un recall de 0.8824 y un AUC de 0.9201. Estas métricas indican que el modelo tuvo una buena capacidad general para clasificar pacientes. Sin embargo, en un problema médico, la exactitud por sí sola no es suficiente. La razón es que los errores no tienen el mismo costo clínico. Un falso positivo puede llevar a exámenes adicionales o preocupación innecesaria, pero un falso negativo puede implicar que un paciente enfermo sea clasificado como sano. Por eso, en este proyecto se prestó especial atención al recall o sensibilidad, que mide la capacidad del modelo para detectar correctamente a los pacientes con enfermedad cardíaca.

La matriz de confusión complementa esta interpretación. El modelo identificó correctamente 90 pacientes con enfermedad cardíaca y 65 pacientes sin enfermedad. Además, generó 17 falsos positivos y 12 falsos negativos. Desde la perspectiva clínica, los 12 falsos negativos son el resultado más delicado, porque representan pacientes enfermos que el modelo no detectó. Aunque ningún modelo es perfecto, el número de falsos negativos fue relativamente bajo en comparación con los verdaderos positivos. Esto sugiere que el modelo puede ser útil como herramienta de apoyo para priorizar pacientes, siempre que no se utilice como diagnóstico definitivo.

El AUC de 0.9201 refuerza la calidad del Random Forest. Esta métrica mide la capacidad del modelo para separar las clases positiva y negativa a lo largo de diferentes umbrales de decisión. Un AUC superior a 0.90 indica una capacidad discriminativa excelente. En una aplicación práctica, esto sería valioso porque permitiría ajustar el umbral según el objetivo clínico. Por ejemplo, si el objetivo fuera reducir al máximo los falsos negativos, se podría mover el umbral para aumentar sensibilidad, aunque eso implique aceptar más falsos positivos.

La importancia de variables también fue coherente con el contexto clínico. La variable más importante fue `cp`, correspondiente al tipo de dolor torácico. Este resultado tiene sentido porque el dolor torácico es una señal clave en la evaluación cardiovascular. También aparecieron como variables importantes `chol`, `thalach`, `oldpeak`, `age` y `exang`. Estas variables representan colesterol, frecuencia cardíaca máxima, depresión ST, edad y angina inducida por ejercicio. Es decir, el modelo no se apoyó principalmente en variables irrelevantes, sino en factores clínicos razonablemente asociados con riesgo cardíaco. Esto fortalece la credibilidad del modelo y facilita la comunicación de resultados.

En contraste, el primer MLPRegressor no logró buenos resultados al intentar predecir `chol`. El modelo obtuvo MAE de 43.4284, RMSE de 55.3219 y R² de -0.1298. Además, al compararlo con un modelo base que simplemente predice el promedio del colesterol, el MLP fue inferior. Esto indica que el modelo no encontró relaciones suficientemente fuertes entre las variables disponibles y el colesterol. Este resultado no debe interpretarse como un fracaso del proyecto, sino como un hallazgo analítico importante. El colesterol depende de factores que no están completamente representados en el dataset, como dieta, genética, peso corporal, medicamentos, hábitos de vida, triglicéridos y antecedentes familiares.

Este punto evidencia una idea central en analítica de negocios y Machine Learning: la calidad y pertinencia de las variables puede ser más importante que la complejidad del algoritmo. Una red neuronal puede modelar relaciones no lineales, pero no puede aprender patrones que no están presentes en los datos. Por eso, aunque el MLPRegressor es un modelo más flexible que una regresión lineal simple, no consiguió explicar adecuadamente el colesterol con las variables disponibles.

A partir de este hallazgo, se realizó un cambio metodológico: se reemplazó la variable objetivo `chol` por `thalach`, que corresponde a la frecuencia cardíaca máxima alcanzada. Esta decisión tuvo una justificación clínica y estadística. La frecuencia cardíaca máxima está más relacionada con variables presentes en el dataset, como edad, dolor torácico, angina inducida por ejercicio, oldpeak y target. Al realizar este cambio, el MLPRegressor sí logró superar al modelo base. El MAE bajó a 18.6394, el RMSE a 23.7781 y el R² subió a 0.1779. Aunque el R² no es alto, el resultado indica que el modelo sí aprendió parte de la variabilidad de la variable objetivo.

Este contraste entre `chol` y `thalach` es uno de los hallazgos más valiosos del proyecto. Muestra que la selección de la variable objetivo tiene un impacto directo sobre el desempeño de los modelos de pronóstico. También demuestra que el dataset es más fuerte para tareas de clasificación de enfermedad cardíaca que para pronósticos clínicos complejos. La clasificación aprovecha patrones combinados entre variables clínicas y target, mientras que el pronóstico exige que la variable numérica esté suficientemente explicada por el resto de las columnas.

En términos de toma de decisiones, el Random Forest es el modelo más útil del proyecto. Permite clasificar pacientes y priorizar atención, además de ofrecer interpretabilidad mediante importancia de variables. El MLPRegressor final aporta valor como análisis complementario, pero no debería ser utilizado como modelo principal. Su utilidad está en mostrar cómo la selección del target puede mejorar o deteriorar el desempeño predictivo.

En conclusión, el proyecto demuestra que los modelos de Machine Learning pueden apoyar la toma de decisiones en salud, pero también evidencia sus límites. Un modelo con buenas métricas puede ser útil, pero siempre debe evaluarse con cuidado, especialmente en contextos clínicos. La interpretación de resultados, la revisión de falsos negativos, la calidad de los datos y la supervisión profesional son elementos indispensables para cualquier implementación real.

# 12. Consideraciones éticas y prácticas

El uso de Machine Learning en salud exige especial cuidado. Los modelos pueden apoyar la toma de decisiones, pero no deben sustituir el juicio clínico de un profesional médico.

## Riesgos principales

1. **Falsos negativos:** son especialmente críticos porque pueden clasificar como sano a un paciente realmente enfermo.
2. **Sesgos del dataset:** los datos provienen de instituciones específicas y pueden no representar a todas las poblaciones.
3. **Privacidad:** los datos médicos son sensibles y deben manejarse con anonimización, seguridad y consentimiento adecuado.
4. **Generalización limitada:** un modelo entrenado con datos históricos puede no funcionar igual en otra institución o país.
5. **Uso inadecuado:** el modelo debe verse como apoyo, no como diagnóstico final.

## Principio de uso responsable

El modelo puede servir como herramienta de priorización y apoyo analítico, pero cualquier decisión médica debe ser revisada por personal de salud calificado.

# 13. Conclusiones

1. El dataset `heart_disease_combined.csv` contiene 920 registros y 15 columnas, con variables clínicas útiles para el análisis de enfermedad cardíaca.
2. Random Forest fue el modelo más sólido del proyecto, con Accuracy 0.8424, F1 Score 0.8612, Recall 0.8824 y AUC 0.9201.
3. La matriz de confusión mostró que el modelo detectó correctamente 90 pacientes enfermos y 65 pacientes sanos, aunque presentó 12 falsos negativos.
4. Las variables más relevantes fueron `cp`, `chol`, `thalach`, `oldpeak`, `age` y `exang`, todas coherentes con criterios cardiovasculares.
5. El primer MLPRegressor para `chol` no superó el modelo base, lo que muestra limitaciones del dataset para explicar el colesterol.
6. El cambio de variable objetivo a `thalach` mejoró el desempeño del MLPRegressor y permitió superar el baseline.
7. El proyecto cumple con la aplicación de un modelo de clasificación y un modelo de pronóstico, utilizando métricas apropiadas para cada caso.
8. Machine Learning puede apoyar decisiones médicas, pero requiere validación externa, control ético y supervisión profesional.

# 14. Recomendaciones

- Mantener Random Forest como modelo principal para clasificación de enfermedad cardíaca.
- Utilizar MLPRegressor como análisis complementario, especialmente para evaluar variables objetivo numéricas con mayor coherencia clínica.
- No utilizar el modelo en decisiones médicas reales sin validación externa.
- Buscar datasets más completos con variables como peso, IMC, dieta, medicamentos, tabaquismo, antecedentes familiares y biomarcadores.
- Evaluar modelos adicionales de regresión como RandomForestRegressor, GradientBoostingRegressor o Ridge Regression.
- Incorporar técnicas de explicabilidad como SHAP en futuras versiones del proyecto.
- Ajustar umbrales de clasificación si se busca priorizar reducción de falsos negativos.
- Documentar el proyecto en GitHub con notebook, dataset, presentación y README.

# 15. Bibliografía

- Janosi, A., Steinbrunn, W., Pfisterer, M., & Detrano, R. (1988). *Heart Disease*. UCI Machine Learning Repository. DOI: https://doi.org/10.24432/C52P4X  
- Kaggle. *UCI Heart Disease Dataset – Combined Version*. Archivo `heart_disease_combined.csv`.  
- Scikit-learn. *RandomForestClassifier documentation*.  
- Scikit-learn. *MLPRegressor documentation*.  
- Scikit-learn. *GridSearchCV documentation*.  
- Scikit-learn. *ROC Curve, AUC and permutation importance documentation*.  
- Material del curso Analítica de Negocios – Universidad EAFIT.

# 16. Anexo: estructura sugerida para GitHub

La rúbrica solicita que el portafolio se aloje en GitHub. Se recomienda crear un repositorio con la siguiente estructura:

```text
Heart-Disease-ML-Analytics/
│
├── README.md
├── Proyecto_Final_Heart_Disease.ipynb
├── heart_disease_combined.csv
├── presentacion/
│   └── presentacion_final.pdf
└── images/
    └── graficas_exportadas.png
```

## Contenido recomendado del README.md

- Nombre del proyecto.
- Objetivo general.
- Dataset utilizado.
- Modelos implementados.
- Métricas principales.
- Conclusiones.
- Integrantes.
- Link al notebook en Google Colab.

## Forma de entregar

1. Subir el notebook `.ipynb` a GitHub.
2. Subir el dataset o indicar claramente cómo acceder a él.
3. Compartir el enlace del repositorio en el buzón del reto.
4. Compartir también el enlace del Google Colab si el profesor lo solicita.